# Project MissingNo — training on Kaggle

Runs `training/ppo.py` from the GitHub repository on Kaggle's hardware: the GPU does the learning, the CPU cores run the emulators.

**One-time setup**
1. **Settings → Accelerator → GPU** (T4 ×2 or P100).
2. **Settings → Internet → On** (requires a verified phone number on your Kaggle account).
3. **Add Input → your private dataset** with the ROMs and start states of the games you train on: `pokemon_red.gb` + `red_start.state`, and for several games also `pokemon_blue.gb` + `blue_start.state`, `pokemon_yellow.gbc` + `yellow_start.state`; for FireRed `pokemon_firered.gba` + `firered_start.state` + `mgba_libretro.so` (see docs/SETUP.md). Keep it **private**: ROMs must never be shared.

**Long runs:** use **Save Version → Save & Run All (Commit)**. The notebook runs in the background, even with your PC off, and everything in `/kaggle/working` is kept as the version's output.

**Resuming a run in a new session:** Add Input → *Your Work* → this notebook (the previous version's output), keep the same `RUN_NAME`, and set `RESUME = True`. The notebook finds the last checkpoint by itself. Alternatively, put `latest.pt` (and the `exploration_*.pkl` files next to it) in a private dataset. The archive and the curriculum are saved next to the checkpoints (`exploration_<game>.pkl`) and continue after a resume; the self-imitation buffer is rebuilt from scratch.

In [ ]:
# ---------------------------------------------------------------- configuration
REPO = "https://github.com/simoninoProgrammatore/project-MissingNo.git"
BRANCH = "main"

RUN_NAME = "ppo_s1"          # same name across sessions of the same run
GAMES = "red"                # one game, or several in one model: "red,blue,yellow"
SEED = 1
TOTAL_STEPS = 100_000_000    # the whole run, across all sessions
TIME_LIMIT_HOURS = 11.0      # stop cleanly before Kaggle's session limit
RESUME = False               # True: continue RUN_NAME from the previous session's output
NUM_ENVS = None              # None = one game per CPU core (rounded to a multiple of the games)
RUN_SPEED_TEST = True        # quick speed diagnosis before training (~1 minute)
EXTRA_ARGS = []              # e.g. ["--ent-coef", "0.02"]

# Example: one model, the first badge of Red, Blue and Yellow, stopping when all three
# are won (see docs/SETUP.md). v2.2 ends episodes stuck in a loop by itself.
# RUN_NAME, GAMES, TOTAL_STEPS = "gen1_s1", "red,blue,yellow", 400_000_000
# EXTRA_ARGS = ["--reward-version", "v2.2", "--episode-steps", "30000",
#               "--archive-prob", "0.3", "--curriculum-prob", "0.3", "--sil-coef", "1.0",
#               "--record-every", "8", "--stop-at-goal"]
#
# Example: the "v3" model, for the whole game (GPU needed: about 6.6M parameters):
# RUN_NAME = "red_v3_s1"
# EXTRA_ARGS = ["--episode-steps", "100000", "--archive-prob", "0.3", "--curriculum-prob", "0.3",
#               "--sil-coef", "1.0", "--record-every", "8", "--stop-at-goal",
#               "--downscale", "1", "--network", "impala", "--memory", "gru",
#               "--gamma", "0.999", "--norm-rewards"]
#
# Example: the same run with a short-term memory (compare it with the one without):
# RUN_NAME = "badge_gru_s1"
# EXTRA_ARGS = [...same as above..., "--memory", "gru"]

## 1. Hardware

In [ ]:
import os, shutil, subprocess
CPUS = os.cpu_count()
print(f"CPU cores: {CPUS}")
# nvidia-smi exists only on GPU machines: a CPU-only session is fine for the small network.
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout)
else:
    print("No GPU: training runs on the CPU (fine for the small network; the v3 model needs a GPU).")


## 2. Code and dependencies

In [ ]:
# The code goes in /tmp so that it does not end up in the notebook's output.
REPO_DIR = "/tmp/missingno"
import re, shutil, subprocess
shutil.rmtree(REPO_DIR, ignore_errors=True)
clone = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO, REPO_DIR],
                       capture_output=True, text=True)
if clone.returncode != 0:
    raise RuntimeError(f"Could not download the code from {REPO}:\n{clone.stderr}\n"
                       "Check REPO and BRANCH in the configuration cell, and that Internet is on.")
!cd {REPO_DIR} && git log -1 --format="Code version: %h %s"

# PyTorch is already installed on Kaggle (with GPU support): we only add what is missing.
# PyBoy and Gymnasium are pinned to the exact versions of uv.lock, i.e. the ones used on
# your PC: emulator save states (red_start.state) are NOT compatible across PyBoy versions.
lock = open(f"{REPO_DIR}/uv.lock").read()
def locked(name):
    return re.search(rf'\[\[package\]\]\nname = "{name}"\nversion = "([^"]+)"', lock).group(1)
PINS = f"pyboy=={locked('pyboy')} gymnasium=={locked('gymnasium')}"
print("Installing", PINS)
!pip install -q {PINS} tensorboard

# Our packages are used straight from the source folders, nothing to build.
SRC = ":".join(f"{REPO_DIR}/packages/{p}/src" for p in ["core", "games", "envs", "agents"])
os.environ["PYTHONPATH"] = SRC + ":" + os.environ.get("PYTHONPATH", "")
!python --version

## 3. ROM, start state and outputs

In [ ]:
from pathlib import Path
import shutil

GAME_LIST = [g.strip() for g in GAMES.split(",") if g.strip()]

def find(*names):
    for name in names:
        hits = list(Path("/kaggle/input").rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{' or '.join(names)} not found in /kaggle/input: add your private dataset as input")

for game in GAME_LIST:
    rom = find(f"pokemon_{game}.gb", f"pokemon_{game}.gbc", f"pokemon_{game}.gba")
    # The start state may have either name in the dataset; the code wants <game>_start.state.
    state = find(f"{game}_start.state", f"pokemon_{game}.state", f"{game}.state")
    shutil.copy(rom, f"{REPO_DIR}/roms/{rom.name}")
    shutil.copy(state, f"{REPO_DIR}/states/{game}_start.state")
    print(f"{game}: {rom.name}, {state.name} -> states/{game}_start.state")

# Game Boy Advance games need the mGBA core: the one in your dataset (same version as the
# one that made your start state), otherwise downloaded.
gba_games = [g for g in GAME_LIST if list(Path("/kaggle/input").rglob(f"pokemon_{g}.gba"))]
if gba_games:
    Path(f"{REPO_DIR}/cores").mkdir(exist_ok=True)
    cores = list(Path("/kaggle/input").rglob("mgba_libretro.so"))
    if cores:
        shutil.copy(cores[0], f"{REPO_DIR}/cores/mgba_libretro.so")
        print(f"mGBA core: {cores[0]}")
    else:
        print("No mgba_libretro.so in the dataset: downloading one (it may not load your start state).")
        !cd {REPO_DIR} && python scripts/get_mgba_core.py

# Runs are written to /kaggle/working/runs, which Kaggle keeps as output.
OUT = Path("/kaggle/working/runs")
OUT.mkdir(parents=True, exist_ok=True)
link = Path(REPO_DIR) / "runs"
if not link.exists():
    link.symlink_to(OUT)
print(f"Runs will be saved in {OUT}")

## 4. Resume (only if `RESUME = True`)

In [ ]:
resume_args = []
if RESUME:
    # The previous checkpoint: a previous version's output added as input (the whole run
    # folder), and/or latest.pt (plus exploration_*.pkl, if you have them) in a dataset.
    # If there are several, the one with the most training steps wins.
    import torch
    found = list(Path("/kaggle/input").rglob("latest.pt"))
    if not found:
        raise FileNotFoundError(
            f"No latest.pt in the inputs: add the previous version's output of {RUN_NAME}, "
            "or a dataset with latest.pt."
        )
    def steps(p):
        return torch.load(p, map_location="cpu", weights_only=False).get("global_step", 0)
    ranked = sorted(((steps(p), p) for p in found), key=lambda t: t[0], reverse=True)
    for s, p in ranked:
        print(f"  found {p} ({s:,} steps)")
    latest = ranked[0][1]
    if latest.parent.name == "checkpoints" and latest.parent.parent.name == RUN_NAME:
        shutil.copytree(latest.parent.parent, OUT / RUN_NAME, dirs_exist_ok=True)  # curves too
    else:
        dest = OUT / RUN_NAME / "checkpoints"
        dest.mkdir(parents=True, exist_ok=True)
        for f in [latest, *latest.parent.glob("exploration_*.pkl")]:
            shutil.copy(f, dest / f.name)
    resume_args = ["--resume", f"runs/{RUN_NAME}/checkpoints/latest.pt"]
    kept = sorted(p.name for p in (OUT / RUN_NAME / "checkpoints").glob("exploration_*.pkl"))
    print(f"Resuming {RUN_NAME} from {latest} ({ranked[0][0]:,} steps)")
    print(f"Exploration (archive, curriculum): {kept or 'none, it starts empty'}")
else:
    print("Starting a new run.")


## 5. Speed test (optional)

In [ ]:
if RUN_SPEED_TEST:
    envs = sorted({2, max(1, CPUS // 2), CPUS})
    !cd {REPO_DIR} && python scripts/diagnose_speed.py --envs {" ".join(map(str, envs))} --seconds 5 --torch-threads 1 2>&1 | grep -v -i "warn\|sdl"

## 6. Training

In [ ]:
import sys
num_envs = NUM_ENVS or CPUS
if len(GAME_LIST) > 1:
    num_envs = max(len(GAME_LIST), num_envs // len(GAME_LIST) * len(GAME_LIST))
cmd = [
    sys.executable, "training/ppo.py",
    "--run-name", RUN_NAME,
    "--seed", str(SEED),
    *(["--games", ",".join(GAME_LIST)] if len(GAME_LIST) > 1 else ["--game", GAME_LIST[0]]),
    "--total-steps", str(TOTAL_STEPS),
    "--num-envs", str(num_envs),
    "--torch-threads", "1",
    "--device", "auto",
    "--time-limit-hours", str(TIME_LIMIT_HOURS),
    *resume_args,
    *EXTRA_ARGS,
]
print(" ".join(cmd[1:]), "\n")

# Stream the output live; skip the noisy library warnings.
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if "warn" not in line.lower() and "sdl" not in line.lower():
        print(line, end="")
print("Exit code:", proc.wait())

## 7. Results

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

acc = EventAccumulator(str(OUT / RUN_NAME), size_guidance={"scalars": 0})
acc.Reload()
tags = acc.Tags()["scalars"]

def series(tag):
    events = acc.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

# One row per game: milestones, tiles, return. Single-game runs have no game in the tags.
multi = len(GAME_LIST) > 1
fig, axes = plt.subplots(len(GAME_LIST), 3, figsize=(18, 4 * len(GAME_LIST)), squeeze=False)
for row, game in zip(axes, GAME_LIST):
    prefix = f"milestones/{game}/" if multi else "milestones/"
    for tag in sorted(t for t in tags if t.startswith(prefix) and t.endswith("_rate")):
        row[0].plot(*series(tag), label=tag.split("/")[-1].replace("_rate", ""))
    row[0].set_title(f"{game}: milestone rate (last 50 episodes)"); row[0].set_ylim(-0.05, 1.05)
    row[0].legend(fontsize=7)
    for name, ax in [("tiles_visited", row[1]), ("return", row[2])]:
        tag = f"episode/{game}/{name}" if multi else f"episode/{name}"
        if tag in tags:
            ax.plot(*series(tag), alpha=0.5); ax.set_title(f"{game}: {name}")
    for ax in row:
        ax.set_xlabel("steps")
plt.tight_layout(); plt.show()

if "charts/SPS" in tags:
    print(f"Last SPS: {series('charts/SPS')[1][-1]:,.0f}")

## 8. Goal replay and GIFs

If the run reached the final goal (`--stop-at-goal`), its replay is in `replays/`: download it and watch the exact game at normal speed on your PC with `scripts/replay.py`. The GIFs of the record-breaking episodes are in `gifs/`.


In [ ]:
run_out = OUT / RUN_NAME
for sub in ("replays", "gifs"):
    files = sorted(f for f in (run_out / sub).rglob("*") if f.is_file()) if (run_out / sub).exists() else []
    print(f"{sub}: {len(files)} file(s)")
    for f in files[-10:]:
        print("  ", f.relative_to(run_out / sub))
ckpt = run_out / "checkpoints"
for f in sorted(ckpt.glob("winner*.pt")):
    print("Goal reached:", f.name)
if (ckpt / "winner.pt").exists():
    print("\nFINAL GOAL REACHED in every game: model in checkpoints/winner.pt")
else:
    print("\nFinal goal not reached in every game yet: resume in a new session (RESUME = True).")

## 9. Download

Everything in `/kaggle/working/runs/<RUN_NAME>` (curves, checkpoints, config) is in the version's **Output** tab.
Download it, put it in `runs/` on your PC, then:

```
uv run tensorboard --logdir runs
uv run python scripts/watch.py --checkpoint runs/<RUN_NAME>/checkpoints/latest.pt
```